In [ ]:

from __future__ import annotations
import os, sys, json, time, math, random, hashlib, platform, subprocess, importlib.util
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import OrderedDict
from contextlib import nullcontext
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_erosion, distance_transform_edt, generate_binary_structure
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_SETTINGS = {
    "data_root": "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation",
    "output_dir": "/kaggle/working/emcad_brats" if Path("/kaggle").exists() else "/content/emcad_brats",
    "source_dir": "/kaggle/working/emcad_official" if Path("/kaggle").exists() else "/content/emcad_official",
    "pretrained_path": "",  
    "allow_download": True, 
    "encoder": "pvt_v2_b0",
    "resume": "",          
    "smoke": False,         
    "workers": 0,           
}
# ===========================================================

COMMIT = '26c9c31f731f749b62c5fe83f44376dac75f3aa8'
SOURCE_HASHES = {
    'lib/pvtv2.py': '9711ee6600e3e5274b927a7aecd706ad459173b1c86867aa56522fad5ec3f5f3',
    'lib/decoders.py': '4b36981117376ea005aed8844f4ea9c846f432286695b85c1530797a20157a18',
    'LICENSE': '2e9976d237bc90bc2e99911f72281670ad8d08aa423e8d0b96d0621f8f04f297',
}
REGIONS = ('ET', 'TC', 'WT')

@dataclass
class Config:
    data_root: str = '/kaggle/input'  
    output_dir: str = '/kaggle/working/emcad_brats'
    source_dir: str = '/kaggle/working/emcad_official'
    encoder: str = 'pvt_v2_b0'  
    pretrained: bool = True
    pretrained_path: str = ''  
    allow_download: bool = True  
    seed: int = 42
    val_fraction: float = 0.20
    epochs: int = 40
    steps_per_epoch: int = 400  
    batch_size: int = 8  
    workers: int = 2
    patient_reuse_steps: int = 8  
    tumor_slice_probability: float = 0.67
    lr: float = 0.0001
    weight_decay: float = 0.01
    warmup_epochs: int = 2
    amp: bool = True
    deterministic: bool = True
    eval_every: int = 2
    eval_batch_size: int = 8
    session_hours: float = 10.5  
    save_every_steps: int = 100
    resume: str = '' 
    eval_only: bool = False
    device: str = 'cuda'
    save_nifti: bool = False
    smoke: bool = False  

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False)); tmp.replace(path)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()


def fetch(url, destination):
    from urllib.request import urlopen, Request
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_suffix(destination.suffix + '.part')
    try:
        with urlopen(Request(url, headers={'User-Agent': 'EMCAD-BraTS-research'}), timeout=90) as r, open(tmp, 'wb') as f:
            import shutil
            shutil.copyfileobj(r, f)
        tmp.replace(destination)
    finally:
        tmp.unlink(missing_ok=True)


def official_sources(cfg):
   
    root = Path(cfg.source_dir)
    for name, expected in SOURCE_HASHES.items():
        p = root / name
        if not p.exists():
            if not cfg.allow_download:
                raise FileNotFoundError(f'Offline: attach official source {name}, set source_dir. Commit={COMMIT}')
            fetch(f'https://raw.githubusercontent.com/SLDGroup/EMCAD/{COMMIT}/{name}', p)
        if sha256(p) != expected:
            raise RuntimeError(f'Upstream source hash mismatch: {p}. Use pinned commit {COMMIT}.')
    mods = []
    for name in ('pvtv2', 'decoders'):
        spec = importlib.util.spec_from_file_location(f'brats_official_{name}', root/'lib'/f'{name}.py')
        key = f'brats_official_{name}'
        if key in sys.modules:
            mod = sys.modules[key]
        else:
            mod = importlib.util.module_from_spec(spec); sys.modules[key] = mod
            spec.loader.exec_module(mod)
        mods.append(mod)
    return mods


class BraTSEMCAD(nn.Module):
    def __init__(self, cfg, initialize_pretrained=True):
        super().__init__()
        pvt, dec = official_sources(cfg)
        if cfg.encoder not in ('pvt_v2_b0', 'pvt_v2_b2'):
            raise ValueError('Supported encoders: pvt_v2_b0, pvt_v2_b2')
        self.backbone = getattr(pvt, cfg.encoder)()
        self.pretrained_info = {'used': False}
        if cfg.pretrained and initialize_pretrained:
            path = Path(cfg.pretrained_path) if cfg.pretrained_path else Path(cfg.source_dir)/f'{cfg.encoder}.pth'
            if not path.exists():
                if not cfg.allow_download or cfg.pretrained_path:
                    raise FileNotFoundError(f'Pretrained weights missing: {path}. Attach official PVT weights or explicitly set pretrained=False.')
                fetch(f'https://github.com/whai362/PVT/releases/download/v2/{cfg.encoder}.pth', path)
            sd = torch.load(path, map_location='cpu', weights_only=True)
            if 'state_dict' in sd: sd = sd['state_dict']
            if 'model' in sd: sd = sd['model']
            # Official ImageNet classifier is unused. All encoder tensors must match.
            sd = {k.removeprefix('module.'): v for k, v in sd.items()}
            sd = {k: v for k, v in sd.items() if not k.startswith('head.')}
            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
            if [k for k in missing if not k.startswith('head.')] or unexpected:
                raise RuntimeError(f'Wrong encoder checkpoint: missing={missing}, unexpected={unexpected}')
            self.pretrained_info = {'used': True, 'sha256': sha256(path), 'file': path.name}
        # Constructors in upstream ignore in_chans kwargs: replace the actual projection.
        old = self.backbone.patch_embed1.proj
        new = nn.Conv2d(4, old.out_channels, old.kernel_size, old.stride, old.padding, bias=old.bias is not None)
        if self.pretrained_info['used']:
            # Inflate RGB filters to 4 modalities, preserving summed response for equal channels.
            with torch.no_grad():
                new.weight.copy_(old.weight.mean(1, keepdim=True).repeat(1,4,1,1) * (3.0/4.0))
                if old.bias is not None: new.bias.copy_(old.bias)
        self.backbone.patch_embed1.proj = new
        if hasattr(self.backbone, 'head'): self.backbone.head = nn.Identity()
        channels = [256,160,64,32] if cfg.encoder.endswith('b0') else [512,320,128,64]
        self.decoder = dec.EMCAD(channels=channels, kernel_sizes=[1,3,5], expansion_factor=2,
                                 dw_parallel=True, add=True, lgag_ks=3, activation='relu')
        self.heads = nn.ModuleList([nn.Conv2d(c, 4, 1) for c in channels])

    def forward(self, x):
        # Native 240x240 slices become 256x256; never resize labels or drop edges.
        h, w = x.shape[-2:]
        x = F.pad(x, (0, (-w)%32, 0, (-h)%32))
        if min(x.shape[-2:]) < 32: raise ValueError('Image dimensions must be >=32')
        f = self.backbone(x)
        features = self.decoder(f[-1], [f[2],f[1],f[0]])
        heads = [F.interpolate(head(y), size=x.shape[-2:], mode='bilinear', align_corners=False)[...,:h,:w]
                 for head,y in zip(self.heads,features)]
        return heads


def fused_logits(heads):
    return torch.stack(heads).sum(0)


def dice_ce(logits, target):
    logits = logits.float()
    ce = F.cross_entropy(logits, target)
    prob = logits.softmax(1)
    truth = F.one_hot(target,4).permute(0,3,1,2).float()
    dims = (0,2,3)
    dice = (2*(prob*truth).sum(dims)+1e-5)/(prob.sum(dims)+truth.sum(dims)+1e-5)
    # Exclude background. Empty classes still receive false-positive gradients.
    return .5*ce + .5*(1-dice[1:].mean())


def loss_fn(heads, y):
    # Main fused prediction plus auxiliary supervision of every decoder scale.
    return dice_ce(fused_logits(heads),y) + .5*sum(dice_ce(h,y) for h in heads)/len(heads)


def discover(root):
    root = Path(root)
    if not root.exists(): raise FileNotFoundError(f'Data root missing: {root}')
    cases = []; unlabeled = []
    t1s = sorted(list(root.rglob('*_t1.nii.gz')) + list(root.rglob('*_t1.nii')))
    for t1 in t1s:
        suffix = '_t1.nii.gz' if t1.name.endswith('.nii.gz') else '_t1.nii'
        name = t1.name[:-len(suffix)]
        paths = {}
        for mod in ('t1','t1ce','t2','flair'):
            matches = [t1.parent/f'{name}_{mod}{ext}' for ext in ('.nii.gz','.nii')]
            matches = [p for p in matches if p.exists()]
            if len(matches)!=1: raise ValueError(f'{name}: expected one {mod} volume beside {t1}; found {matches}')
            paths[mod] = str(matches[0])
        segs = [t1.parent/f'{name}_seg{ext}' for ext in ('.nii.gz','.nii')]
        segs = [p for p in segs if p.exists()]
        if not segs:
            # Some 2020 mirrors contain a misnamed segmentation. Accept only one
            # otherwise unassigned NIfTI with verified discrete BraTS labels.
            other = [p for p in t1.parent.iterdir() if (p.name.endswith('.nii') or p.name.endswith('.nii.gz')) and str(p) not in paths.values()]
            if len(other)==1:
                labels = np.unique(nib.load(other[0]).get_fdata(dtype=np.float32))
                if set(labels.tolist()).issubset({0.,1.,2.,4.}) and np.any(labels>0):
                    segs=other; print(f'Explicit filename recovery: {name} segmentation -> {other[0].name}',flush=True)
        if not segs:
            unlabeled.append(name); continue
        if len(segs)!=1: raise ValueError(f'{name}: ambiguous segmentation files {segs}')
        cases.append({'id': name, **paths, 'seg': str(segs[0])})
    if unlabeled:
        print(f'Excluded {len(unlabeled)} unlabeled patients (not in the internal split): {unlabeled}',flush=True)
    if not cases:
        raise FileNotFoundError('No *_seg.nii[.gz] found. Attach/extract LABELED BraTS 2020/2021 training NIfTI data (not PNG/HDF5 or official unlabeled validation).')
    ids = [c['id'] for c in cases]
    if len(ids)!=len(set(ids)):
        raise ValueError('Duplicate patient IDs found; point data_root at ONE dataset copy/year to avoid leakage.')
    years = {year for year in ('2020','2021') if any(year in n for n in ids)}
    if len(years)>1: raise ValueError('Do not mix BraTS 2020 and 2021: overlapping subjects can leak across splits.')
    if len(cases)<6: raise ValueError(f'Need >=6 labeled patients; found {len(cases)}')
    return cases


def read_case(c):
    
    ref = nib.as_closest_canonical(nib.load(c['t1']))
    if len(ref.shape)!=3: raise ValueError(f'{c["id"]}: expected 3D NIfTI')
    aff = ref.affine
    spacing = np.asarray(nib.affines.voxel_sizes(aff),dtype=float)
    directions = aff[:3,:3]/spacing
    if not np.allclose(directions.T@directions,np.eye(3),atol=1e-3):
        raise ValueError(f'{c["id"]}: sheared grid; resample before physical HD95')
    if not np.allclose(directions,np.eye(3),atol=1e-3):
        raise ValueError(f'{c["id"]}: oblique grid; use standard aligned BraTS volumes for axial slicing')
    images=[]
    for mod in ('t1','t1ce','t2','flair'):
        im = nib.as_closest_canonical(nib.load(c[mod]))
        if im.shape!=ref.shape or not np.allclose(im.affine,aff,atol=1e-3):
            raise ValueError(f'{c["id"]}: {mod} not co-registered')
        a = im.get_fdata(dtype=np.float32)
        if not np.isfinite(a).all(): raise ValueError(f'{c["id"]}: nonfinite MRI')
        mask = a!=0
        if mask.sum()<10: raise ValueError(f'{c["id"]}: empty modality {mod}')
        vals=a[mask]; lo,hi=np.percentile(vals,[.5,99.5]); vals=np.clip(vals,lo,hi)
        a[mask]=(vals-vals.mean())/max(float(vals.std()),1e-6)
        a[~mask]=0
        images.append(a)
    sg = nib.as_closest_canonical(nib.load(c['seg']))
    if sg.shape!=ref.shape or not np.allclose(sg.affine,aff,atol=1e-3):
        raise ValueError(f'{c["id"]}: segmentation grid mismatch')
    raw = sg.get_fdata(dtype=np.float32)
    labels=np.unique(raw)
    if not set(labels.tolist()).issubset({0.,1.,2.,4.}):
        raise ValueError(f'{c["id"]}: expected raw BraTS labels 0,1,2,4; got {labels}')
    y=raw.astype(np.uint8); y[y==4]=3
    return {'x':np.stack(images).astype(np.float16), 'y':y,
            'spacing':spacing.tolist(),'affine':aff, 'id':c['id']}


def make_split(cases,cfg, previous=None):
    lookup={c['id']:c for c in cases}
    if previous is not None:
        if not set(previous['train']+previous['val']).issubset(lookup):
            raise ValueError('Resume dataset is missing patients from the saved split')
        split=previous
    else:
        ids=sorted(lookup); rng=np.random.default_rng(cfg.seed); rng.shuffle(ids)
        nv=max(2,round(len(ids)*cfg.val_fraction)); nv=min(nv,len(ids)-2)
        split={'train':sorted(ids[nv:]),'val':sorted(ids[:nv]), 'seed':cfg.seed,
               'kind':'internal patient-level holdout of labeled training data'}
        if cfg.smoke: split['train']=split['train'][:4]; split['val']=split['val'][:2]
    assert not set(split['train'])&set(split['val'])
    return [lookup[i] for i in split['train']],[lookup[i] for i in split['val']],split


class BatchDataset(Dataset):
    """One item = one batch from two distinct patients. Stateless seeded sampling.
    LRU holds two float16 volumes per worker, avoiding a 20-80GB disk cache.
    Each budget epoch permutes patients and cycles through adjacent patient pairs.
    """
    def __init__(self,cases,cfg,epoch,start_step=0):
        self.cases=cases; self.cfg=cfg; self.epoch=epoch; self.start=start_step
        self.order=np.random.default_rng(cfg.seed+epoch).permutation(len(cases))
        self.cache=OrderedDict()
    def __len__(self): return self.cfg.steps_per_epoch-self.start
    def get_case(self,index):
        if index not in self.cache:
            self.cache[index]=read_case(self.cases[index])
            while len(self.cache)>2: self.cache.popitem(last=False)
        self.cache.move_to_end(index)
        return self.cache[index]
    def __getitem__(self,index):
        step=index+self.start
        rng=np.random.default_rng(np.random.SeedSequence([self.cfg.seed,self.epoch,step]))
        group=step//self.cfg.patient_reuse_steps
        patient_indices=[self.order[(2*group+j)%len(self.order)] for j in range(2)]
        xs=[]; ys=[]; ids=[]
        for idx in patient_indices:
            a=self.get_case(int(idx)); tumor=np.flatnonzero((a['y']>0).any(axis=(0,1)))
            for _ in range(self.cfg.batch_size//2):
                z=int(rng.choice(tumor)) if len(tumor) and rng.random()<self.cfg.tumor_slice_probability else int(rng.integers(a['y'].shape[2]))
                x=a['x'][...,z].astype(np.float32); y=a['y'][...,z].astype(np.int64)
                if rng.random()<.5: x=x[:,::-1,:]; y=y[::-1,:]
                if rng.random()<.5: x=x[:,:,::-1]; y=y[:,::-1]
                # No rotations needed; preserves rectangular native grids too.
                x=x.copy(); y=y.copy()
                if rng.random()<.8:
                    nonzero=x!=0
                    x=x*rng.uniform(.9,1.1,(4,1,1)).astype(np.float32)
                    x=x+rng.normal(0,.05,x.shape).astype(np.float32)*nonzero
                xs.append(x); ys.append(y); ids.append(a['id'])
        shapes={x.shape for x in xs}
        if len(shapes)!=1: raise ValueError('Mixed in-plane sizes; use standard BraTS aligned dataset')
        return {'x':torch.from_numpy(np.stack(xs)), 'y':torch.from_numpy(np.stack(ys)), 'ids':ids}


def worker_init(_):
    torch.set_num_threads(1)


def regions(y):
    return (y==3, (y==1)|(y==3), y>0)


def region_metrics(pred,truth,spacing):
   
    pred=np.asarray(pred,dtype=bool); truth=np.asarray(truth,dtype=bool)
    n,m=int(pred.sum()),int(truth.sum())
    if n+m==0: return 1.,0.,'both_empty'
    penalty=float(np.linalg.norm((np.array(pred.shape)-1)*np.asarray(spacing)))
    if n==0 or m==0: return 0.,penalty,'prediction_empty' if n==0 else 'reference_empty'
    dice=2*int(np.count_nonzero(pred&truth))/(n+m)
    conn=generate_binary_structure(3,1)
    ps=pred^binary_erosion(pred,structure=conn,border_value=0)
    gs=truth^binary_erosion(truth,structure=conn,border_value=0)
    d1=distance_transform_edt(~gs,sampling=spacing)[ps]
    d2=distance_transform_edt(~ps,sampling=spacing)[gs]
    hd=max(float(np.percentile(d1,95)),float(np.percentile(d2,95)))
    return float(dice),hd,'nonempty'


def autocast(cfg,device):
    return torch.autocast('cuda',dtype=torch.float16) if cfg.amp and device.type=='cuda' else nullcontext()


def sync(device):
    if device.type=='cuda': torch.cuda.synchronize(device)


@torch.inference_mode()
def predict_case(model,a,cfg,device):
    model.eval(); h,w,z=a['y'].shape; pred=np.empty((h,w,z),np.uint8)
    for lo in range(0,z,cfg.eval_batch_size):
        hi=min(lo+cfg.eval_batch_size,z)
        x=torch.from_numpy(a['x'][...,lo:hi].transpose(3,0,1,2).astype(np.float32)).to(device)
        with autocast(cfg,device): logits=fused_logits(model(x))
        if not torch.isfinite(logits).all(): raise FloatingPointError('Nonfinite validation logits; no metric will be fabricated')
        pred[...,lo:hi]=logits.argmax(1).cpu().numpy().transpose(1,2,0)
    return pred


def overlay(a,pred,path):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    y=a['y']; area=(y>0).sum(axis=(0,1)); positive=np.flatnonzero(area)
    # Predeclared GT-based selection, never select on prediction quality.
    zz=list(dict.fromkeys([int(positive[len(positive)//4]),int(np.argmax(area)),int(positive[3*len(positive)//4])])) if len(positive) else [y.shape[2]//2]
    fig,axes=plt.subplots(len(zz),3,figsize=(10,3.4*len(zz)),squeeze=False)
    cmap=ListedColormap(['black','#f4d35e','#35c875','#f34d60'])
    for row,z in enumerate(zz):
        base=a['x'][3,:,:,z].T
        for col,title in enumerate(['FLAIR','Ground truth','Prediction']):
            ax=axes[row,col]; ax.imshow(base,cmap='gray',origin='lower',vmin=-1,vmax=3)
            if col:
                mask=(y if col==1 else pred)[:,:,z].T
                ax.imshow(np.ma.masked_equal(mask,0),cmap=cmap,vmin=0,vmax=3,alpha=.55,origin='lower',interpolation='nearest')
            ax.set_title(f'{title} | axial z={z}'); ax.axis('off')
    fig.suptitle(a['id']+' | canonical RAS display')
    fig.legend(handles=[Patch(color=cmap(i),label=s) for i,s in [(1,'NCR/NET'),(2,'Edema'),(3,'Enhancing tumor')]],loc='lower center',ncol=3)
    fig.tight_layout(rect=[0,.05,1,.95]); fig.savefig(path,dpi=160); plt.close(fig)


def summarize(rows):
    report={'metric_definition':'3D patient macro-average; HD95=max of directed 95th percentiles (mm)',
            'empty_policy':'both empty: Dice=1, HD95=0; one empty: Dice=0, HD95=physical grid diagonal',
            'regions':{}}
    for name in REGIONS:
        rr=[r for r in rows if r['region']==name]
        d=np.array([r['dice'] for r in rr]); h=np.array([r['hd95_mm'] for r in rr])
        finite=[r['hd95_mm'] for r in rr if r['status']=='nonempty']
        report['regions'][name]={'dice_mean':float(d.mean()),'dice_std':float(d.std()),
              'hd95_mm_mean':float(h.mean()),'hd95_mm_std':float(h.std()),
              'hd95_mm_median':float(np.median(h)), 'n':len(rr),
              'empty_counts':{s:sum(r['status']==s for r in rr) for s in ['both_empty','prediction_empty','reference_empty']},
              'hd95_nonempty_only_mean':float(np.mean(finite)) if finite else None}
    report['mean_dice']=float(np.mean([r['dice'] for r in rows]))
    report['mean_hd95_mm']=float(np.mean([r['hd95_mm'] for r in rows]))
    return report


def evaluate(model,cases,cfg,device,out=None,compute_hd=True):
    rows=[]; started=time.perf_counter(); inference_seconds=0.
    if out: Path(out).mkdir(parents=True,exist_ok=True)
    for i,c in enumerate(cases):
        a=read_case(c); sync(device); t=time.perf_counter()
        pred=predict_case(model,a,cfg,device); sync(device); inference_seconds+=time.perf_counter()-t
        for name,p,g in zip(REGIONS,regions(pred),regions(a['y'])):
            if compute_hd: dice,hd,status=region_metrics(p,g,a['spacing'])
            else:
                den=int(p.sum())+int(g.sum()); dice=2*int((p&g).sum())/den if den else 1.
                hd=0.; status='not_computed'
            rows.append({'patient_id':c['id'],'region':name,'dice':dice,'hd95_mm':hd,'status':status})
        if out:
            if i<6: overlay(a,pred,Path(out)/f'overlay_{i+1:02d}.png')
            if cfg.save_nifti:
                # Export canonical prediction back into the original NIfTI voxel orientation.
                source=nib.load(c['t1']); canonical=nib.orientations.axcodes2ornt(('R','A','S'))
                orig=nib.orientations.io_orientation(source.affine)
                back=nib.orientations.ornt_transform(canonical,orig)
                labels=pred.copy(); labels[labels==3]=4
                labels=nib.orientations.apply_orientation(labels,back).astype(np.uint8)
                header=source.header.copy(); header.set_data_dtype(np.uint8)
                nib.save(nib.Nifti1Image(labels,source.affine,header),Path(out)/f'{c["id"]}_prediction.nii.gz')
        if (i+1)%10==0 or i==len(cases)-1: print(f'  validation {i+1}/{len(cases)}',flush=True)
    summary=summarize(rows) if compute_hd else {'mean_dice':float(np.mean([r['dice'] for r in rows]))}
    summary.update({'validation_wall_seconds':time.perf_counter()-started,'inference_seconds':inference_seconds,
                    'patients':len(cases),'split':'internal validation, not hidden challenge validation'})
    if out:
        import csv
        with open(Path(out)/'per_patient_metrics.csv','w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
        write_json(Path(out)/'summary.json',summary)
    return summary


def write_report(out,cfg,env,final,perf,history):
    import csv
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    lines=['# EMCAD: 2D four-modality BraTS segmentation', '',
        '**SMOKE TEST ONLY — not a final experiment.**' if cfg.smoke else 'Single-run internal validation experiment; no hidden test-set or challenge-server claim.', '',
        '## Method', '',
        f'Encoder: {cfg.encoder}. Official EMCAD decoder commit: {COMMIT}. Four native MRI channels (T1, T1ce, T2, FLAIR); native axial slices padded to multiples of 32, no spatial resizing. MSCAM, LGAG and EUCB are retained from the upstream implementation.', '',
        'Four mutually exclusive classes: background, NCR/NET, edema, enhancing tumor. ET={3}; TC={1,3}; WT={1,2,3} after remapping raw label 4 to 3. Each of four decoder heads emits four logits. Their sum is the fused prediction. Loss is fused Dice+CE plus 0.5 times the mean auxiliary Dice+CE loss (each Dice+CE weights its components 0.5/0.5). This is a documented adaptation of the training objective, not an exact reproduction of the original training recipe.', '',
        '## Budget and selection', '',
        f"Seed {cfg.seed}; {env['train_patient_count']} train / {env['validation_patient_count']} validation patients. Planned {cfg.epochs} budget epochs × {cfg.steps_per_epoch} optimizer steps × {cfg.batch_size} slices = {env['planned_slice_draws']} slice draws. Actual completed optimizer steps: {perf['global_optimizer_steps']}. Best checkpoint selected by mean full-volume ET/TC/WT Dice on the same internal validation split.", '',
        f'AdamW lr={cfg.lr}, weight_decay={cfg.weight_decay}; {cfg.warmup_epochs}-epoch linear warmup then cosine decay; gradient clipping norm=1; AMP={cfg.amp}; 2 patients/batch, tumor-slice draw probability={cfg.tumor_slice_probability}. Each budget epoch re-permutes patients; it is not a complete slice pass. training_exposure.json records actual patient coverage.', '',
        f"GPU: {env['gpu']}; torch={env['torch']}; encoder pretrained={env['pretraining']}. Parameters={env['parameters']:,}.", '',
        '## Validation', '',
        '| Region | Dice mean | Dice SD | HD95 mean (mm) | HD95 median (mm) |',
        '|---|---:|---:|---:|---:|']
    rows=[]
    for name,r in final['regions'].items():
        lines.append(f"| {name} | {r['dice_mean']:.4f} | {r['dice_std']:.4f} | {r['hd95_mm_mean']:.3f} | {r['hd95_mm_median']:.3f} |")
        rows.append({'region':name,'dice_mean':r['dice_mean'],'dice_std':r['dice_std'],'hd95_mean_mm':r['hd95_mm_mean'],'hd95_median_mm':r['hd95_mm_median']})
    lines += [f"| Macro mean | {final['mean_dice']:.4f} | — | {final['mean_hd95_mm']:.3f} | — |", '',
        'Metrics are computed per reconstructed 3D patient volume at original spacing, then macro-averaged. HD95 uses 6-connected voxel surfaces and max of the two directed 95th percentiles. Both-empty masks receive Dice=1 and HD95=0; exactly one empty mask receives Dice=0 and the physical grid diagonal as HD95. This is an explicitly defined local convention; do not compare these HD95 values directly with a challenge server using a different empty-mask or percentile convention. Empty-case counts and nonempty-only HD95 are in validation/summary.json.', '',
        '## Measured performance', '', '```json', json.dumps(perf,indent=2), '```', '',
        'Peak VRAM is PyTorch allocator memory for this process/session, not nvidia-smi board usage. Final evaluation runs with training optimizer buffers still present, so its memory is not a stripped inference-only deployment footprint. Timings exclude validation/checkpoint writes; session wall time includes them. Soft time guard is checked between training steps; validation can extend beyond it.', '',
        '## Limitations', '',
        'One patient-level holdout and one seed; validation is used for checkpoint selection and is not an unbiased test estimate. A 2D model does not explicitly use through-plane context. No TTA, connected-component filtering, or ET-size threshold was tuned. Actual runtime and accuracy are hardware/data dependent. This run does not establish a new state of the art.', '',
        '## Visual evidence', '',
        'First six validation patient IDs in sorted split order, selected before inspecting results. Within each case: lower/upper tumor quartile slices and maximal tumor-area slice. Colors: yellow NCR/NET, green edema, red ET; TC is yellow+red, WT is all colors.', '']
    for path in sorted((Path(out)/'validation').glob('overlay_*.png')):
        lines.append(f'![{path.stem}](validation/{path.name})')
    lines += ['', '## Sources', '',
        '- EMCAD: https://github.com/SLDGroup/EMCAD',
        '- Paper: https://openaccess.thecvf.com/content/CVPR2024/html/Rahman_EMCAD_Efficient_Multi-scale_Convolutional_Attention_Decoding_for_Medical_Image_Segmentation_CVPR_2024_paper.html',
        '- BraTS labels/evaluation: https://www.med.upenn.edu/cbica/brats2020/tasks.html',
        '- BraTS data: https://www.med.upenn.edu/cbica/brats2020/data.html',
        '- PVT weights: https://github.com/whai362/PVT/releases/tag/v2']
    (Path(out)/'REPORT.md').write_text('\n'.join(lines))
    with open(Path(out)/'validation'/'region_summary.csv','w',newline='') as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
    fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    hl=[r for r in history if r['train_loss_this_session'] is not None]
    hv=[r for r in history if r['validation_mean_dice'] is not None]
    ax[0].plot([r['global_step'] for r in hl],[r['train_loss_this_session'] for r in hl],marker='o')
    ax[0].set_title('Training loss (epoch/session mean)')
    ax[1].plot([r['global_step'] for r in hv],[r['validation_mean_dice'] for r in hv],marker='o')
    ax[1].set_title('Full-volume validation mean Dice'); ax[1].set_ylim(0,1)
    for a in ax: a.set_xlabel('Optimizer step'); a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(Path(out)/'learning_curves.png',dpi=150); plt.close(fig)


def seed_all(cfg):
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark=not cfg.deterministic
    torch.backends.cudnn.deterministic=cfg.deterministic
    torch.use_deterministic_algorithms(cfg.deterministic,warn_only=True)


def get_rng():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'].cpu())
    if s['cuda'] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all([x.cpu() for x in s['cuda']])


def save_checkpoint(path,obj):
    path=Path(path); tmp=path.with_suffix('.tmp'); torch.save(obj,tmp); tmp.replace(path)


def run(cfg=Config()):
    if cfg.smoke:
        cfg.epochs=1; cfg.steps_per_epoch=2; cfg.eval_every=1; cfg.pretrained=False
        cfg.workers=0; cfg.batch_size=2
    if cfg.batch_size<2 or cfg.batch_size%2: raise ValueError('batch_size must be even and >=2')
    if cfg.steps_per_epoch<1 or cfg.epochs<1 or cfg.eval_every<1 or cfg.patient_reuse_steps<1 or cfg.save_every_steps<1:
        raise ValueError('Epoch/step/evaluation/save/reuse counts must be positive')
    if not 0<cfg.val_fraction<1 or not 0<=cfg.tumor_slice_probability<=1: raise ValueError('Invalid fractions')
    if cfg.eval_only and not cfg.resume: raise ValueError('eval_only requires resume checkpoint')
    out=Path(cfg.output_dir); out.mkdir(parents=True,exist_ok=True)
    if not cfg.resume and (out/'last.pt').exists(): raise FileExistsError('Existing last.pt: set resume or choose a fresh output_dir')
    seed_all(cfg); device=torch.device(cfg.device)
    if device.type=='cuda' and not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU accelerator. CPU is only for synthetic tests.')
    # Checkpoints use pickle for optimizer/RNG; only load your own trusted checkpoints.
    previous=torch.load(cfg.resume,map_location='cpu',weights_only=False) if cfg.resume else None
    if previous:
        keys=['encoder','seed','batch_size','steps_per_epoch','patient_reuse_steps','tumor_slice_probability','lr','weight_decay','epochs','warmup_epochs','amp','smoke']
        if not cfg.eval_only:
            for k in keys:
                if previous['config'][k]!=getattr(cfg,k): raise ValueError(f'Resume mismatch: {k}. Keep training protocol fixed; start a new run for changes.')
        else:
            if cfg.encoder!=previous['config']['encoder']: raise ValueError('Set encoder to checkpoint encoder')
            # Reports for evaluation must describe the checkpoint's real training protocol.
            for k in keys: setattr(cfg,k,previous['config'][k])
    cases=discover(cfg.data_root)
    train,val,split=make_split(cases,cfg,previous['split'] if previous else None)
    write_json(out/'split.json',split); write_json(out/'config.json',asdict(cfg))
    write_json(out/'manifest.json',cases)
    print(f'{len(train)} train / {len(val)} validation PATIENTS. Modalities=T1,T1ce,T2,FLAIR. Labels 4->3.',flush=True)
    # Early inspection detects wrong data/grid/label assumptions before allocating the model.
    inspect=read_case(train[0]); print('First case:',inspect['x'].shape,'spacing',inspect['spacing'],flush=True); del inspect
    model=BraTSEMCAD(cfg,initialize_pretrained=previous is None).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.amp and device.type=='cuda',init_scale=1024.)
    history=[]; global_step=0; epoch0=0; step0=0; best_score=-1.; timings=[]; exposure={}; cumulative_train_seconds=0.
    provenance=model.pretrained_info
    if previous:
        model.load_state_dict(previous['model']); provenance=previous['pretrained_info']
        history=previous['history']; best_score=previous['best_score']; global_step=previous['global_step']
        epoch0=previous['epoch']; step0=previous['step_in_epoch']; exposure=previous.get('exposure',{})
        cumulative_train_seconds=previous.get('cumulative_train_seconds',0.)
        timings=previous.get('timings',[])
        if not cfg.eval_only:
            opt.load_state_dict(previous['optimizer']); scaler.load_state_dict(previous['scaler']); set_rng(previous['rng'])
        # Each checkpoint carries the selected best weights, so last.pt is portable across sessions.
        best_weights=previous.get('best_model')
    else: best_weights=None
    env={'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__,'nibabel':nib.__version__,
         'cuda_runtime':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'device':str(device),
         'gpu':torch.cuda.get_device_name(device) if device.type=='cuda' else None,
         'gpu_total_vram_gib':torch.cuda.get_device_properties(device).total_memory/2**30 if device.type=='cuda' else None,
         'upstream_commit':COMMIT,'upstream_hashes':SOURCE_HASHES,'pretraining':provenance,
         'parameters':sum(p.numel() for p in model.parameters()),'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
         'validation_patient_count':len(val),'train_patient_count':len(train),
         'planned_optimizer_steps':cfg.epochs*cfg.steps_per_epoch,'planned_slice_draws':cfg.epochs*cfg.steps_per_epoch*cfg.batch_size,
         'padded_size':'ceil(native X/32)*32 by ceil(native Y/32)*32; 240 becomes 256',
         'determinism':'seeded; deterministic algorithms warn_only; bitwise equality across environments not guaranteed'}
    write_json(out/'environment.json',env)
    (out/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    if step0 == cfg.steps_per_epoch:
        epoch0 += 1; step0 = 0
    session_start=time.perf_counter(); stopped=False; peak_train_alloc=peak_train_reserved=0
    def checkpoint(epoch,step):
        return {'model':model.state_dict(),'optimizer':opt.state_dict(),'scaler':scaler.state_dict(),
                'config':asdict(cfg),'split':split,'epoch':epoch,'step_in_epoch':step,'global_step':global_step,
                'best_score':best_score,'best_model':best_weights,'history':history,'rng':get_rng(),
                'pretrained_info':provenance,'timings':timings,'exposure':exposure,
                'cumulative_train_seconds':cumulative_train_seconds}
    if not cfg.eval_only:
        for epoch in range(epoch0,cfg.epochs):
            ds=BatchDataset(train,cfg,epoch,start_step=step0)
            # Explicit generator prevents loader construction consuming model/dropout RNG on resume.
            loader=DataLoader(ds,batch_size=None,num_workers=cfg.workers,pin_memory=device.type=='cuda',
                worker_init_fn=worker_init,generator=torch.Generator().manual_seed(cfg.seed+epoch),
                **({'prefetch_factor':2} if cfg.workers else {}))
            model.train(); losses=[]
            if device.type=='cuda': torch.cuda.reset_peak_memory_stats(device)
            iterator=iter(loader)
            for step in range(step0,cfg.steps_per_epoch):
                if (time.perf_counter()-session_start)>cfg.session_hours*3600:
                    save_checkpoint(out/'last.pt',checkpoint(epoch,step))
                    if global_step == 0: raise RuntimeError('Time budget expired before a training step. Resume last.pt with a larger session_hours.')
                    stopped=True; break
                tick=time.perf_counter(); batch=next(iterator); sync(device); gpu_start=time.perf_counter()
                x=batch['x'].to(device,non_blocking=True); y=batch['y'].to(device,non_blocking=True)
                warmup=max(1,cfg.warmup_epochs*cfg.steps_per_epoch)
                total=cfg.epochs*cfg.steps_per_epoch
                scale=(global_step+1)/warmup if global_step<warmup else .5*(1+math.cos(math.pi*min(1.,(global_step-warmup)/max(1,total-warmup))))
                for pg in opt.param_groups: pg['lr']=cfg.lr*scale
                # Retry finite-loss / overflowing-gradient AMP steps with a lower scale.
                # No optimizer step or sampled batch is silently skipped.
                for amp_retry in range(8):
                    opt.zero_grad(set_to_none=True)
                    with autocast(cfg,device): loss=loss_fn(model(x),y)
                    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite training loss. Last periodic checkpoint remains usable; no silent skipping.')
                    scaler.scale(loss).backward(); scaler.unscale_(opt)
                    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=False)
                    if torch.isfinite(norm):
                        scaler.step(opt); scaler.update(); break
                    if not scaler.is_enabled() or amp_retry==7:
                        raise FloatingPointError('Nonfinite gradients after retries; inspect inputs/checkpoint, or start a full-precision run.')
                    scaler.update(new_scale=max(scaler.get_scale()/2.,1.))
                    print(f'AMP overflow: retrying same batch at scale {scaler.get_scale()}',flush=True)
                sync(device)
                end=time.perf_counter(); global_step+=1; losses.append(float(loss.detach()))
                timings.append({'global_step':global_step,'compute_seconds':end-gpu_start,'end_to_end_seconds':end-tick,'amp_retries':amp_retry})
                cumulative_train_seconds+=end-tick
                for pid in batch['ids']: exposure[pid]=exposure.get(pid,0)+1
                if global_step%20==0: print(f'epoch {epoch+1}/{cfg.epochs} step {step+1}/{cfg.steps_per_epoch} loss={np.mean(losses):.4f} compute={end-gpu_start:.3f}s',flush=True)
                if (step+1)%cfg.save_every_steps==0:
                    save_checkpoint(out/'last.pt',checkpoint(epoch,step+1))
            del iterator,loader,ds
            if device.type=='cuda':
                peak_train_alloc=max(peak_train_alloc,torch.cuda.max_memory_allocated(device))
                peak_train_reserved=max(peak_train_reserved,torch.cuda.max_memory_reserved(device))
            # Validate after each configured interval and on a time-guard stop.
            score=None
            if stopped or (epoch+1)%cfg.eval_every==0 or epoch+1==cfg.epochs:
                print('Full-volume validation for model selection...',flush=True)
                result=evaluate(model,val,cfg,device,compute_hd=False); score=result['mean_dice']
                if score>best_score:
                    best_score=score; best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                    save_checkpoint(out/'best.pt',checkpoint(epoch,step if stopped else cfg.steps_per_epoch))
                print(f'Validation mean ET/TC/WT Dice={score:.4f}; best={best_score:.4f}',flush=True)
            history.append({'epoch':epoch+1,'completed':not stopped,'train_loss_this_session':float(np.mean(losses)) if losses else None,'validation_mean_dice':score,'global_step':global_step})
            write_json(out/'history.json',history)
            save_checkpoint(out/'last.pt',checkpoint(epoch,step if stopped else cfg.steps_per_epoch))
            if stopped: break
            step0=0
        # best.pt is selected solely on validation Dice; HD95 never tunes the model.
        if best_weights is None: raise RuntimeError('No best model: training ended before validation')
        model.load_state_dict(best_weights)
    elif best_weights is not None:
        model.load_state_dict(best_weights)
    if device.type=='cuda': torch.cuda.reset_peak_memory_stats(device)
    final=evaluate(model,val,cfg,device,out=out/'validation',compute_hd=True)
    elapsed=time.perf_counter()-session_start
    valid_times=[t for t in timings if t['global_step']>10] or timings
    perf={'global_optimizer_steps':global_step,'cumulative_train_step_seconds':cumulative_train_seconds,
          'current_session_seconds_including_validation':elapsed,'stopped_by_soft_time_guard':stopped,
          'timing_scope':'synchronized compute includes host-to-device, forward, loss, backward, optimizer; end-to-end adds batch fetch; excludes checkpoint/validation',
          'warmup_exclusion':'first 10 global training steps if available',
          'train_peak_allocated_gib_current_session':peak_train_alloc/2**30 if device.type=='cuda' and not cfg.eval_only else None,
          'train_peak_reserved_gib_current_session':peak_train_reserved/2**30 if device.type=='cuda' and not cfg.eval_only else None,
          'final_eval_peak_allocated_gib':torch.cuda.max_memory_allocated(device)/2**30 if device.type=='cuda' else None,
          'sampled_train_patients':len(exposure),'total_train_patients':len(train),
          'selected_validation_mean_dice':best_score,'smoke_only':cfg.smoke}
    for field in ['compute_seconds','end_to_end_seconds']:
        vals=[t[field] for t in valid_times]
        perf[field]={'mean':float(np.mean(vals)) if vals else None,'median':float(np.median(vals)) if vals else None,'p95':float(np.percentile(vals,95)) if vals else None}
    write_json(out/'performance.json',perf); write_json(out/'training_exposure.json',exposure)
    write_json(out/'step_timings.json',timings)
    write_report(out,cfg,env,final,perf,history)
    print(json.dumps(final,indent=2)); print(json.dumps(perf,indent=2))
    print(f'Artifacts: {out}. Real results must come from this run; smoke metrics are not reportable.')
    return final,perf


def main(argv=None):
    """Notebook cells ignore kernel flags; a Python CLI still accepts --config."""
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', help='JSON file with Config fields')
    in_notebook = ('ipykernel' in sys.modules or 'google.colab' in sys.modules
                   or Path(sys.argv[0]).name in ('colab_kernel_launcher.py', 'ipykernel_launcher.py'))
    # Do not feed Jupyter's injected -f kernel.json into our application parser.
    args = parser.parse_args([] if argv is None and in_notebook else argv)
    settings = json.loads(Path(args.config).read_text()) if args.config else dict(NOTEBOOK_SETTINGS)
    for key in ('data_root', 'output_dir', 'source_dir', 'pretrained_path', 'resume'):
        if str(settings.get(key, '')).startswith(('https://', 'http://')):
            raise ValueError(f'{key} requires a LOCAL path, not a website link. Attach the Kaggle dataset, then use /kaggle/input/<dataset-folder>/...')
    cfg = Config(**settings)
    print('Notebook-safe startup. Data directory:', cfg.data_root, flush=True)
    return run(cfg)


if __name__ == '__main__':
    main()


Notebook-safe startup. Data directory: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation
Explicit filename recovery: BraTS20_Training_355 segmentation -> W39_1998.09.19_Segm.nii
Excluded 125 unlabeled patients (not in the internal split): ['BraTS20_Validation_001', 'BraTS20_Validation_002', 'BraTS20_Validation_003', 'BraTS20_Validation_004', 'BraTS20_Validation_005', 'BraTS20_Validation_006', 'BraTS20_Validation_007', 'BraTS20_Validation_008', 'BraTS20_Validation_009', 'BraTS20_Validation_010', 'BraTS20_Validation_011', 'BraTS20_Validation_012', 'BraTS20_Validation_013', 'BraTS20_Validation_014', 'BraTS20_Validation_015', 'BraTS20_Validation_016', 'BraTS20_Validation_017', 'BraTS20_Validation_018', 'BraTS20_Validation_019', 'BraTS20_Validation_020', 'BraTS20_Validation_021', 'BraTS20_Validation_022', 'BraTS20_Validation_023', 'BraTS20_Validation_024', 'BraTS20_Validation_025', 'BraTS20_Validation_026', 'BraTS20_Validation_027', 'BraTS20_Validation_028', 'BraTS20_Valida

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/kaggle/working/emcad_official/lib/pvtv2.py:385: UserWarning: Overwriting pvt_v2_b0 in registry with brats_official_pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/emcad_official/lib/pvtv2.py:395: UserWarning: Overwriting pvt_v2_b1 in registry with brats_official_pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an

epoch 1/40 step 20/400 loss=2.0215 compute=0.111s
epoch 1/40 step 40/400 loss=1.9609 compute=0.112s
epoch 1/40 step 60/400 loss=1.8694 compute=0.113s
epoch 1/40 step 80/400 loss=1.7481 compute=0.110s
epoch 1/40 step 100/400 loss=1.6226 compute=0.110s
epoch 1/40 step 120/400 loss=1.5237 compute=0.113s
epoch 1/40 step 140/400 loss=1.4420 compute=0.115s
epoch 1/40 step 160/400 loss=1.3786 compute=0.111s
epoch 1/40 step 180/400 loss=1.3280 compute=0.113s
epoch 1/40 step 200/400 loss=1.2833 compute=0.112s
epoch 1/40 step 220/400 loss=1.2393 compute=0.112s
epoch 1/40 step 240/400 loss=1.2031 compute=0.112s
epoch 1/40 step 260/400 loss=1.1741 compute=0.109s
epoch 1/40 step 280/400 loss=1.1453 compute=0.113s
epoch 1/40 step 300/400 loss=1.1191 compute=0.113s
epoch 1/40 step 320/400 loss=1.0953 compute=0.113s
epoch 1/40 step 340/400 loss=1.0758 compute=0.133s
epoch 1/40 step 360/400 loss=1.0535 compute=0.111s
epoch 1/40 step 380/400 loss=1.0325 compute=0.111s
epoch 1/40 step 400/400 loss=1.0114